# 09 — The app's pipeline, end to end, and what the study concludes

Notebooks 03 to 08 compared methods. This one runs **the app's own code**,
`kwak_analytics.forecast.run` then `kwak_core.forecast.assemble` (spec §4.14), exactly as
`kwak_api.services.forecast` calls them, on synthetic households whose last 12 months are held
out: the pipeline sees 36 months, chooses a model per category by its own back-test, builds its
conformal bands, and forecasts the 12 months it never saw. Then the synthesis.

In [1]:
import time
import uuid
import warnings
from decimal import Decimal

import numpy as np
import pandas as pd
import torch
from kwak_analytics.forecast import ORDER, SeriesInput, run
from kwak_core.budget import Month
from kwak_core.categories import CategoryKind
from kwak_core.forecast import HORIZON, SEASONAL_HISTORY, Series, assemble, cuts

import study

warnings.filterwarnings("ignore")
df, _ = study.full_panel()
lengths = df.groupby("household")["t"].max() + 1
households = list(np.random.default_rng(3).choice(lengths[lengths >= 48].index.to_numpy(), 30, replace=False))
KIND = {"expense": CategoryKind.EXPENSE, "savings": CategoryKind.SAVINGS, "income": CategoryKind.INCOME}
D = lambda x: Decimal(str(round(float(x), 2)))  # noqa: E731

def household_run(h: int):
    g = df[df.household == h].sort_values("ds")
    months = sorted(g["ds"].unique())[-48:]
    hist, fut = months[:36], months[36:]
    series, known, truth, ids = {}, {}, {}, {}
    for cat, c in g.groupby("category"):
        c = c.set_index("ds")
        key = uuid.uuid5(uuid.NAMESPACE_URL, f"{h}/{cat}")
        ids[key] = cat
        total = [D(c.loc[m, "y"]) for m in hist]
        fixed = [D(c.loc[m, "fixed"]) for m in hist]
        nz = [i for i, x in enumerate(total) if x != 0]
        series[key] = Series(KIND[c["kind"].iloc[0]], tuple(total), tuple(fixed), tuple(fixed), nz[0] if nz else None)
        known[key] = tuple(D(c.loc[m, "fixed"]) for m in fut)  # the fixed expenses due, known in advance
        truth[key] = np.array([c.loc[m, "y"] for m in fut], dtype=float)
    inputs = [SeriesInput(k, s.variable[s.start:], s.start, s.length >= SEASONAL_HISTORY)
              for k, s in series.items() if s.start is not None]
    first = Month(pd.Timestamp(hist[0]).year, pd.Timestamp(hist[0]).month)
    res = run(inputs, first=first, n=36, cuts=cuts(36))
    fc = assemble(series, known, res.backtests, res.futures, ORDER)
    return series, known, truth, ids, fc, res

t = time.perf_counter()
runs = {h: household_run(h) for h in households}
print(f"{len(runs)} households in {time.perf_counter() - t:.0f} s")

30 households in 96 s


## What the pipeline chose

In [2]:
chosen = pd.Series([cf.choice.model if cf.choice else "no lines" for *_, fc, _ in runs.values()
                    for cf in fc.categories.values()])
contest = pd.Series([cf.choice.contest for *_, fc, _ in runs.values() for cf in fc.categories.values() if cf.choice])
print(f"kept the baseline without a contest: {(~contest).mean():.0%} of the series with lines")
chosen.value_counts().to_frame("series")

kept the baseline without a contest: 19% of the series with lines


,series
baseline,238
seasonal_naive,96
average_all,23
ses,11
ets,5
average_12,5
trees,4
average_3,4
naive,2
adida,2


## Accuracy on the 12 held-out months

Each category's forecast (known fixed part plus the chosen model's variable part) against what
happened, the MASE scaled by the category's month-to-month change over its 36 months; against
the same known part plus the 6-month mean, ETS alone, or Chronos-2 (zero-shot) for the variable
part.

In [3]:
from chronos import BaseChronosPipeline
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

device = "cuda" if torch.cuda.is_available() else "cpu"
chronos2 = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=device)
rows = []
for h, (series, known, truth, ids, fc, _) in runs.items():
    keys = [k for k, s in series.items() if s.start is not None]
    var = {k: np.array([float(x) for x in series[k].variable], dtype=float) for k in keys}
    ets = StatsForecast(models=[AutoETS(season_length=12)], freq="MS").forecast(
        df=pd.concat([pd.DataFrame({"unique_id": str(k), "ds": pd.date_range("2020-01-01", periods=36, freq="MS"),
                                    "y": var[k]}) for k in keys]), h=12)
    q, _ = chronos2.predict_quantiles([torch.tensor(var[k], dtype=torch.float32) for k in keys],
                                      prediction_length=12, quantile_levels=[0.5])
    for i, k in enumerate(keys):
        s = series[k]
        tot = np.array([float(x) for x in s.total])
        scale = np.abs(np.diff(tot)).mean()
        if scale == 0:
            continue
        kn = np.array([float(x) for x in known[k]])
        floor = s.kind is not CategoryKind.SAVINGS
        clip = (lambda a: np.maximum(a, 0)) if floor else (lambda a: a)
        preds = {
            "the app's pipeline": np.array([float(b.point) for b in fc.categories[k].bands]),
            "known + Mean6": clip(kn + var[k][-6:].mean()),
            "known + ETS": clip(kn + ets.loc[ets.unique_id == str(k), "AutoETS"].to_numpy()),
            "known + Chronos-2": clip(kn + q[i][0, :, 0].cpu().numpy()),
        }
        for name, p in preds.items():
            rows.append({"household": h, "category": ids[k], "method": name,
                         "mase": np.abs(truth[k] - p).mean() / scale,
                         "ae": np.abs(truth[k] - p).sum(), "y": np.abs(truth[k]).sum()})
acc = pd.DataFrame(rows)
g = acc.groupby("method")
pd.DataFrame({"MASE": g["mase"].mean(), "MASE median": g["mase"].median(), "WAPE": g["ae"].sum() / g["y"].sum()}).sort_values("MASE").round(3)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

,MASE,MASE median,WAPE
method,,,
known + Chronos-2,0.718,0.633,0.083
known + ETS,0.799,0.664,0.088
the app's pipeline,0.822,0.739,0.094
known + Mean6,0.918,0.809,0.145


## Do the app's bands hold on months it never saw?

In [4]:
cov = []
for h, (series, known, truth, ids, fc, _) in runs.items():
    for k, cf in fc.categories.items():
        for i, b in enumerate(cf.bands):
            if b.low != b.high:
                cov.append({"level": "category", "inside": float(b.low) <= truth[k][i] <= float(b.high), "step": i + 1})
    tot = {name: np.zeros(12) for name in ("income", "spent", "saved")}
    for k, s in series.items():
        name = {CategoryKind.INCOME: "income", CategoryKind.EXPENSE: "spent", CategoryKind.SAVINGS: "saved"}[s.kind]
        tot[name] += truth[k]
    tot["rest"] = tot["income"] - tot["spent"] - tot["saved"]
    for name, values in tot.items():
        for i, b in enumerate(getattr(fc, name)):
            if b.low != b.high:
                cov.append({"level": name, "inside": float(b.low) <= values[i] <= float(b.high), "step": i + 1})
cov = pd.DataFrame(cov)
cov.groupby("level")["inside"].agg(["mean", "size"]).round(2).rename(columns={"mean": "coverage", "size": "months"})

,coverage,months
level,,
category,0.78,3859
income,0.80,360
rest,0.82,360
saved,0.94,215
spent,0.78,360


In [5]:
cov[cov.level == "category"].groupby("step")["inside"].mean().round(2).to_frame("category coverage").T

step,1,2,3,4,5,6,7,8,9,10,11,12
category coverage,0.81,0.73,0.77,0.77,0.77,0.8,0.82,0.82,0.79,0.74,0.74,0.76


## The pipeline on held-out years

- **It works as designed**: on 12 months it never saw, the app's pipeline (MASE 0.822, WAPE
  9.4 %) errs **10 % less** than the known fixed part plus the 6-month mean (0.918, 14.5 %).
  Its bands hold: **78 %** of the categories' months inside the 80 % band (from 73 % to 82 % by
  months ahead), **80 %** for income, 78 % for spending, 82 % for the rest; savings over-cover
  (94 %: their variable part is rare withdrawals).
- **It is cautious**: it keeps the 6-month mean for 61 % of the series (19 % without a contest:
  too few months with a line, or a flat training) and picks the seasonal naive for a quarter;
  the trees 4 times out of 392. **ETS alone, everywhere, does slightly better** (0.799): with one
  household's 23 back-test cuts, the 10 % margin often keeps the baseline where ETS would have
  been right.
- **Chronos-2, zero-shot, is the best by a clear margin** (0.718, WAPE 8.3 %), 13 % below the
  pipeline.

# Synthesis

| Question | Evidence | Notebook |
|---|---|---|
| Is a per-series choice worth it? | Yes: −10 % MASE against always the 6-month mean (48 months), −6 % (30 months), out of sample | 03 |
| Best statistical models | AutoARIMA (0.838) and AutoETS (0.850), tied, ahead of Theta, the intermittent methods and the averages; no model wins every category | 03 |
| The 10 % margin | Costs 1 % with long histories, protects short ones (0.894 against 0.927 with no margin) | 03 |
| The 24-month rule (averages only below) | **Worse** than letting all candidates compete behind the margin (0.946 against 0.894 on 12–18-month trainings) | 03 |
| Machine learning | A LightGBM pooled over 60 households is the best non-foundation model (0.819), but within **one** household, as the app must fit it, it falls below the 6-month mean's rank (0.909); scaling per series matters; recursive beats direct; local trees are worst | 04 |
| Foundation models | TimesFM 2.5 (0.768) and Chronos-2 (0.783) zero-shot are the most accurate of all, need no other household's data, run in about 1 s per household on a CPU; they under-forecast by 3–4 % of the money (their median: about 10 % of the spending; the mean of their deciles −2 to +3 %) | 05, 09 |
| The app's conformal bands | Calibrated: 81 % in the sequential back-test, 78–82 % on held-out years; n ≥ 9 matters (n ≥ 5: 78 %, under-covering); about 30 % wider than ETS's Gaussian band; totals conservative (88 %) | 06, 09 |
| Sharpest calibrated bands | Foundation models' and quantile LightGBM's quantiles (CRPS 0.54–0.56 against 0.68 for conformal ETS) | 06 |
| Explaining a forecast | Averages and ETS explain themselves in words; TreeSHAP on the trees shows the same month last year and the category dominate; the calendar-month feature is redundant with the 12-month lag | 07 |
| "Did my target work?" | Recoverable on average, not for one household: error 8–9 points on effects of 10–25 %, detected half the time, a −8-point bias exposed by the placebo in time; confounded by regression to the mean in real use | 08 |

## What it means for the app (spec §4.14)

Supported by the evidence:

- forecasting only the **variable** part, the fixed expenses known (F-FCT-2);
- the **6-month mean as the baseline**, the rolling-origin **back-test** and the **MASE** to rank
  (F-FCT-4);
- a **margin** before leaving the baseline, 10 % being a sound value;
- **conformal bands** with the ⌈(n+1)q⌉ ranks and **n ≥ 9**, coverage measured sequentially, and
  totals' bands from the summed forecasts' errors (F-FCT-5);
- keeping causal estimates **out** of the page for now (notebook 08).

Contradicted, or worth changing:

1. **The 24-month rule** (F-FCT-3: only the averages compete below 24 months of a series'
   history): on this panel it costs accuracy; the 10 % margin alone already protects short
   series. Consider letting every candidate compete from the first cut, behind the margin (the
   seasonal ones need 13 months of training, which the cuts from the 13th month give).
2. **AutoARIMA** is the best statistical model here and is not among the app's candidates; it
   is cheap enough (20 s for 4,606 series-origins).
3. **Foundation models** earn their place on accuracy (−13 % MASE against the pipeline on
   held-out years for Chronos-2), and their CPU cost is small; what stands in the way is
   packaging (PyTorch and pinned weights in the image) and their downward bias. Since built
   (spec v1.13): Chronos-2 and TimesFM 2.5 as candidates in an optional image, at their mean,
   chosen by the same back-test, and set aside when more optimistic than the baseline.
4. **The per-household trees** are rarely chosen and weak (0.909); they could be dropped for
   simplicity, or kept as a harmless candidate. Their calendar-month feature adds nothing.

## Limits of this study

- The data are **synthetic**: the components (lognormal noise, seasonal profiles, breaks) are
  ours, so models that assume them (ETS) may look better than on real spending, and the real
  household's quirks (a refund booked as a negative, a month entered late) are absent.
- One panel, one seed. ARIMA and ETS cannot be told apart (Nemenyi); the foundation models'
  lead over them is well beyond the critical difference.
- The real history will be short for years: the 30-month results matter more than the 48-month
  ones, and the app's own coverage figure, measured on the household's back-test, is the check
  that will count.